<a href="https://colab.research.google.com/github/AyaAbdElNaem/AI_Tools/blob/main/Reconst%26Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Dynamic Device Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)

class NativesLSTMLayer(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(NativesLSTMLayer, self).__init__()
        self.hidden_dim = hidden_dim
        self.W_x = nn.Linear(input_dim, 4 * hidden_dim, bias=True)
        self.W_h = nn.Linear(hidden_dim, 4 * hidden_dim, bias=False)

    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        h = torch.zeros(batch_size, self.hidden_dim, device=x.device)
        c = torch.zeros(batch_size, self.hidden_dim, device=x.device)
        outputs = []
        for t in range(seq_len):
            x_t = x[:, t, :]
            gates = self.W_x(x_t) + self.W_h(h)
            i_gate, f_gate, c_gate, o_gate = gates.chunk(4, dim=1)

            i_t = torch.exp(torch.clamp(i_gate, -5.0, 5.0))
            f_t = torch.exp(torch.clamp(f_gate, -5.0, 5.0))

            c_tilde = torch.tanh(c_gate)
            c = f_t * c + i_t * c_tilde
            o_t = torch.sigmoid(o_gate)
            h = o_t * torch.tanh(c)
            outputs.append(h.unsqueeze(1))
        return torch.cat(outputs, dim=1)

class JointxLSTMAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super(JointxLSTMAutoencoder, self).__init__()

        # Encoder Stack
        self.encoder_xlstm1 = NativesLSTMLayer(input_dim, 32)
        self.encoder_xlstm2 = NativesLSTMLayer(32, 16)
        self.bottleneck = nn.Linear(16, latent_dim)

        # Decoder Stack (Reconstruction Head)
        self.decoder_xlstm1 = NativesLSTMLayer(latent_dim, 16)
        self.decoder_xlstm2 = NativesLSTMLayer(16, 32)
        self.reconstruct = nn.Linear(32, input_dim)

        # Prediction Head (Supervised branch targeting Normalized Carbon Emission)
        self.predictor_head = nn.Linear(latent_dim, 1)

    def forward(self, x):
        # Pass through Encoder
        encoded = self.encoder_xlstm1(x)
        encoded = self.encoder_xlstm2(encoded)
        latent = self.bottleneck(encoded[:, -1, :])

        # Branch 1: Reconstruct Features X
        decoded_input = latent.unsqueeze(1).repeat(1, x.size(1), 1)
        decoded = self.decoder_xlstm1(decoded_input)
        decoded = self.decoder_xlstm2(decoded)
        reconstructed_output = torch.sigmoid(self.reconstruct(decoded))

        # Branch 2: Predict Carbon Target Y directly from latent features
        predicted_carbon = self.predictor_head(latent)

        return reconstructed_output, predicted_carbon, latent

In [2]:
FILE_PATH = '/content/rural_carbon_dataset.csv'
df = pd.read_csv(FILE_PATH)
df_processed = df.copy()

# Feature Encoding & Circular Months Transformations
crop_encoder = LabelEncoder()
df_processed['Crop_Type_Encoded'] = crop_encoder.fit_transform(df_processed['Crop_Type'])
df_processed['Month_sin'] = np.sin(2 * np.pi * df_processed['Month'] / 12)
df_processed['Month_cos'] = np.cos(2 * np.pi * df_processed['Month'] / 12)
df_processed['Livestock_Total'] = df_processed['Livestock_Cows'] + df_processed['Livestock_Pigs']
df_processed['Energy_per_Area'] = df_processed['Household_Energy_kWh'] / (df_processed['Crop_Area_ha'] + 1)
df_processed['Fertilizer_per_Area'] = df_processed['Fertilizer_Usage_kg'] / (df_processed['Crop_Area_ha'] + 1)

feature_cols = [
    'Month_sin', 'Month_cos', 'Crop_Type_Encoded', 'Crop_Area_ha', 'Livestock_Total',
    'Household_Energy_kWh', 'Renewable_Energy_Fraction', 'Temperature_C',
    'Rainfall_mm', 'Energy_per_Area', 'Fertilizer_per_Area'
]

X = df_processed[feature_cols].values.astype(np.float32)
y = df_processed['Carbon_Emission_tCO2'].values.astype(np.float32).reshape(-1, 1)

# Strict Partitioning
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Unified MinMax Normalization for Inputs X & Target Y
scaler_X = MinMaxScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

scaler_y = MinMaxScaler()
y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled = scaler_y.transform(y_test)

# Reshaping for Sequential 3D Layout
X_train_3d = np.expand_dims(X_train_scaled, axis=1)
X_test_3d = np.expand_dims(X_test_scaled, axis=1)

# Package both X and Y into the Loader for Joint Training
train_dataset = TensorDataset(torch.tensor(X_train_3d), torch.tensor(y_train_scaled))
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

test_inputs_X = torch.tensor(X_test_3d).to(device)
test_targets_y = torch.tensor(y_test_scaled).to(device)

print(f"Data conversion successful. Target (Y) successfully normalized. Active device: {device}")

Data conversion successful. Target (Y) successfully normalized. Active device: cpu


In [3]:
feat_dim = X_train_3d.shape[2]
encoding_dim = 8  # Compressed Bottleneck Dimension

model = JointxLSTMAutoencoder(input_dim=feat_dim, latent_dim=encoding_dim).to(device)
criterion_recon = nn.MSELoss()  # For X reconstruction
criterion_pred = nn.MSELoss()   # For Y prediction
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 100
best_test_loss = float('inf')
patience, patience_counter = 20, 0
alpha = 1.0  # Loss weighting parameter balancing reconstruction vs prediction tasks

print("Commencing Joint Unsupervised-Supervised xLSTM Autoencoder Training...")
print("-" * 75)

for epoch in range(epochs):
    # ==================== TRAINING STAGE ====================
    model.train()
    train_total_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        recon_out, pred_carbon, _ = model(batch_x)

        # Calculate joint Multi-task Losses
        loss_recon = criterion_recon(recon_out, batch_x)
        loss_pred = criterion_pred(pred_carbon, batch_y)
        loss_total = loss_recon + (alpha * loss_pred)

        loss_total.backward()
        optimizer.step()

        train_total_loss += loss_total.item() * batch_x.size(0)
    train_total_loss /= len(train_loader.dataset)

    # ==================== MONITORING STAGE (Validation) ====================
    model.eval()
    test_recon_loss = 0.0
    test_pred_loss = 0.0
    with torch.no_grad():
        test_recon, test_pred, _ = model(test_inputs_X)
        test_recon_loss = criterion_recon(test_recon, test_inputs_X).item()
        test_pred_loss = criterion_pred(test_pred, test_targets_y).item()
        test_total_loss = test_recon_loss + (alpha * test_pred_loss)

    if test_total_loss < best_test_loss:
        best_test_loss = test_total_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_joint_xlstm_ae.pth')
    else:
        patience_counter += 1

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:03d}/{epochs}] -> Train Loss: {train_total_loss:.6f} | Test Recon Loss: {test_recon_loss:.6f} | Test Pred Loss: {test_pred_loss:.6f}")

    if patience_counter >= patience:
        print(f"Early stopping triggered at epoch {epoch+1} to avoid severe Multi-task Overfitting.")
        break

print("-" * 75)
print("Joint Training successfully finalized. Loading optimized weights for evaluation...")

# ==================== CRITICAL EVALUATION ====================
if os.path.exists('best_joint_xlstm_ae.pth'):
    model.load_state_dict(torch.load('best_joint_xlstm_ae.pth'))

model.eval()
with torch.no_grad():
    final_recon, final_pred, final_latent = model(test_inputs_X)

# Flatten targets and predictions to calculate standard metrics
y_true_np = test_targets_y.cpu().numpy().flatten()
y_pred_np = final_pred.cpu().numpy().flatten()

X_true_flat = X_test_3d.reshape(-1, feat_dim)
X_recon_flat = final_recon.cpu().numpy().reshape(-1, feat_dim)

# 1. Reconstruction Performance Evaluation
recon_r2 = r2_score(X_true_flat, X_recon_flat)

# 2. Pure Normalized Prediction Performance Evaluation (Our Main Target)
pred_rmse = np.sqrt(mean_squared_error(y_true_np, y_pred_np))
pred_mae = mean_absolute_error(y_true_np, y_pred_np)
pred_r2 = r2_score(y_true_np, y_pred_np)

print("\n========== Official Joint xLSTM Architecture Evaluation ==========")
print(f"Reconstruction Task Test R² (X Integrity)     = {recon_r2:.6f}")
print(f"Normalized Prediction Task Test R² (Carbon Y) = {pred_r2:.6f}")
print(f"Normalized Prediction Task Test RMSE          = {pred_rmse:.6f}")
print(f"Normalized Prediction Task Test MAE           = {pred_mae:.6f}")
print(f"Conditioned Latent Space Shape                = {final_latent.shape}")
print("==================================================================")

Commencing Joint Unsupervised-Supervised xLSTM Autoencoder Training...
---------------------------------------------------------------------------
Epoch [001/100] -> Train Loss: 0.188263 | Test Recon Loss: 0.093725 | Test Pred Loss: 0.020844
Epoch [010/100] -> Train Loss: 0.085140 | Test Recon Loss: 0.070978 | Test Pred Loss: 0.011268
Epoch [020/100] -> Train Loss: 0.050920 | Test Recon Loss: 0.037724 | Test Pred Loss: 0.011484
Epoch [030/100] -> Train Loss: 0.036183 | Test Recon Loss: 0.024061 | Test Pred Loss: 0.011530
Epoch [040/100] -> Train Loss: 0.029394 | Test Recon Loss: 0.017925 | Test Pred Loss: 0.011405
Epoch [050/100] -> Train Loss: 0.021681 | Test Recon Loss: 0.010628 | Test Pred Loss: 0.011375
Epoch [060/100] -> Train Loss: 0.020335 | Test Recon Loss: 0.009119 | Test Pred Loss: 0.011403
Epoch [070/100] -> Train Loss: 0.019639 | Test Recon Loss: 0.008477 | Test Pred Loss: 0.011556
Epoch [080/100] -> Train Loss: 0.018381 | Test Recon Loss: 0.007357 | Test Pred Loss: 0.01139